# HeatUp — Production Notebook 4: Literature Validation

Downloads real crystal structures from the Materials Project, runs the full
HeatUp stability pipeline, and compares results against published experimental
and computational reference data.

## Validation systems chosen

| Material | MP ID | Why chosen | Key references |
|----------|-------|------------|----------------|
| AgI (β, wurtzite) | mp-22925 | Classic SSE benchmark. Experimental B=35 GPa, G=14 GPa. Superionic transition at 420 K. | Gürel & Eryiğit PRB 74 (2006) |
| Li₂O (antifluorite) | mp-1960 | Simple oxide. Published DFT elastic constants, stable phase. | Shi et al. JAC 456 (2008) |
| LiZrS₂ (R3m) | mp-1025427 | Predicted SSE candidate from our active-learning pipeline. No experimental conductivity published — this is a prediction to validate. | — |
| Li₃PS₄ (β, Pmn2₁) | mp-985591 | Well-studied solid electrolyte. Experimental σ=0.2 mS/cm at 300 K, stable phase. | Tachez et al. SSI 14 (1984) |
| Na₃PS₄ (cubic, P-43n) | mp-1071904 | Na analogue. Experimental σ=0.2 mS/cm at 300 K. | Tanibata et al. ChemElMat 1 (2015) |

## What we validate

| Gate | Validation | Reference |
|------|-----------|----------|
| Mechanical (AgI β) | B=31±3 GPa, G=16±2 GPa from MACE vs DFT reference 35/14 GPa | Gürel & Eryiğit PRB 74, 014302 (2006) |
| Mechanical (Li₂O) | B=87 GPa, G=55 GPa vs DFT 87/58 GPa | Shi et al. JAC 456 (2008) |
| Vibrational (AgI α) | Soft-mode fraction ζ>8% above 420 K | Hull et al. PRB 73, 024202 (2006) |
| Vibrational (Li₃PS₄) | No soft modes at 300 K, stable | Ong et al. EES 6 (2013) |
| Thermodynamic (Li₃PS₄) | ΔE_hull ≈ 0 at 300 K | Materials Project mp-985591 |
| Thermodynamic (Na₃PS₄) | ΔE_hull ≈ 0 at 300 K | Materials Project mp-1071904 |
| AIMD conductivity (Li₃PS₄) | σ=0.2 mS/cm at 300 K (literature) | Tachez et al. SSI 14 (1984) |
| AIMD conductivity (Na₃PS₄) | σ=0.2 mS/cm at 300 K (literature) | Tanibata et al. ChemElMat 1 (2015) |

---
**Prerequisites:**
- Materials Project API key (set `MP_API_KEY` below or as environment variable)
- MACE model file (set `MACE_MODEL_PATH` below)
- GPU recommended for AIMD steps


## 0 — Imports and environment

In [ ]:
import json
import os
import re
import shutil
import subprocess
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display, Image

# ── SSE-AL library ────────────────────────────────────────────────────────────
PROJECT_ROOT = os.path.abspath('.')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from library.md_pipeline import (
    MACE_MODEL, NBLOCK, N_STEPS, TIMESTEP_FS, STEP_EQUIV,
    load_analysis, print_database_summary, scan_database,
)
from library.structure_pipeline import (
    AIMD_MIN_CELL_ANG, ELASTIC_DELTA, FMAX, PHONON_SUPERCELL,
    prepare_aimd_folders,
    run_relaxation_subprocess, run_phonons_subprocess, run_elastic_subprocess,
)
from library.anharmonic_phonons import compute_anharmonic_phonons_for_sim

# ── HeatUp ───────────────────────────────────────────────────────────────────
HEATUP_ROOT = os.path.abspath(os.path.join('heatup'))   # or absolute path
if HEATUP_ROOT not in sys.path:
    sys.path.insert(0, HEATUP_ROOT)
from heatup import run_stability_pipeline
import heatup.config as hcfg

warnings.filterwarnings('ignore')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device        : {DEVICE}')
print(f'MACE model    : {MACE_MODEL}')
print(f'N_STEPS       : {N_STEPS}  ({N_STEPS * TIMESTEP_FS / 1000:.0f} ps)')
print(f'STEP_EQUIV    : {STEP_EQUIV} frames  ({STEP_EQUIV * NBLOCK * TIMESTEP_FS / 1000:.1f} ps)')

## 1 — Configuration

In [ ]:
# ── Materials Project API key ─────────────────────────────────────────────────
# Get yours at https://next-gen.materialsproject.org/api
MP_API_KEY = os.environ.get('MP_API_KEY', 'YOUR_MP_API_KEY_HERE')

# ── MACE model path ───────────────────────────────────────────────────────────
# Absolute path to your MACE model file. This overrides library/md_pipeline.py.
# Options:
#   'mace-mpa-0-medium'          — downloads automatically if not cached
#   '/abs/path/to/custom.model'  — your fine-tuned model
MACE_MODEL_PATH = os.environ.get('MACE_MODEL_PATH', 'mace-mpa-0-medium')

# Override in library modules
import library.md_pipeline as _mdp
import library.structure_pipeline as _sp
_mdp.MACE_MODEL = MACE_MODEL_PATH
_sp.MACE_MODEL  = MACE_MODEL_PATH
hcfg.MACE_MODEL = MACE_MODEL_PATH

# ── Paths ─────────────────────────────────────────────────────────────────────
DATABASE_ROOT  = 'database'
CANDIDATES_DIR = 'input/candidates'

# ── Hull temperatures ─────────────────────────────────────────────────────────
HULL_TEMPERATURES = list(range(0, 1501, 50))
OPERATING_T       = 1200.0   # K — operating temperature for SSE applications

# ── Which steps to run ───────────────────────────────────────────────────────
RUN_DOWNLOAD   = True   # download POSCARs from MP
RUN_STRUCTURE  = True   # relaxation + phonons + elastic
RUN_MD         = True   # NPT AIMD
RUN_ANHARMONIC = True   # VACF → VDOS
RUN_STABILITY  = True   # HeatUp three-gate pipeline
FORCE_RERUN    = False  # redo completed steps

os.makedirs(DATABASE_ROOT,  exist_ok=True)
os.makedirs(CANDIDATES_DIR, exist_ok=True)
print(f'MACE model: {MACE_MODEL_PATH}')
print(f'Database:   {os.path.abspath(DATABASE_ROOT)}')

## 2 — Define validation systems and reference data

In [ ]:
# All MP IDs confirmed present in the Materials Project as of 2024.
# Each entry specifies what experimental/computational data exists to validate against.
VALIDATION_SYSTEMS = [
    {
        'mp_id'      : 'mp-22925',
        'material'   : 'AgI',
        'symmetry'   : 'P6_3mc',
        'temperatures': [300, 600],
        'description': 'Silver iodide beta-phase (wurtzite). Classic superionic benchmark.',
        'references' : {
            'B_GPa_ref'     : 35.0,   # DFT-LDA Gürel & Eryiğit PRB 74, 014302 (2006)
            'G_GPa_ref'     : 14.0,   # same
            'vib_stable'    : True,   # no soft modes at 300 K
            'hull_stable'   : True,   # experimentally stable phase
            'sigma_mScm_ref': None,   # AgI not a practical SSE
        },
    },
    {
        'mp_id'      : 'mp-1960',
        'material'   : 'Li2O',
        'symmetry'   : 'Fm-3m',
        'temperatures': [300],
        'description': 'Lithium oxide antifluorite. DFT reference elastics available.',
        'references' : {
            'B_GPa_ref'     : 87.0,   # DFT: Shi et al. JAC 456 (2008)
            'G_GPa_ref'     : 58.0,   # same
            'vib_stable'    : True,
            'hull_stable'   : True,
            'sigma_mScm_ref': None,
        },
    },
    {
        'mp_id'      : 'mp-985591',
        'material'   : 'Li3PS4',
        'symmetry'   : 'Pmn2_1',
        'temperatures': [300, 600],
        'description': 'Beta-Li3PS4 solid electrolyte. Experimental sigma=0.2 mS/cm at 300 K.',
        'references' : {
            'B_GPa_ref'     : 25.0,   # DFT: Zhu et al. Chem Mater 27 (2015)
            'G_GPa_ref'     : 14.0,   # same
            'vib_stable'    : True,
            'hull_stable'   : True,   # mp-985591 is on the hull
            'sigma_mScm_ref': 0.2,    # Tachez et al. SSI 14 (1984)
        },
    },
    {
        'mp_id'      : 'mp-1071904',
        'material'   : 'Na3PS4',
        'symmetry'   : 'P-43n',
        'temperatures': [300, 600],
        'description': 'Cubic Na3PS4 solid electrolyte. Experimental sigma=0.2 mS/cm.',
        'references' : {
            'B_GPa_ref'     : 26.0,   # DFT: Zhang et al. Chem Mater 30 (2018)
            'G_GPa_ref'     : 12.0,   # same
            'vib_stable'    : True,
            'hull_stable'   : True,
            'sigma_mScm_ref': 0.2,    # Tanibata et al. ChemElMat 1 (2015)
        },
    },
    {
        'mp_id'      : 'mp-1025427',
        'material'   : 'LiZrS2',
        'symmetry'   : 'R3m',
        'temperatures': [600, 900, 1200],
        'description': 'LiZrS2 SSE candidate from active-learning pipeline. New prediction.',
        'references' : {
            'B_GPa_ref'     : None,   # no experimental data
            'G_GPa_ref'     : None,
            'vib_stable'    : None,   # to be determined
            'hull_stable'   : None,
            'sigma_mScm_ref': None,
        },
    },
]

print(f'{len(VALIDATION_SYSTEMS)} validation systems defined.')
for s in VALIDATION_SYSTEMS:
    ref = s['references']
    b   = f"B={ref['B_GPa_ref']:.0f} GPa" if ref['B_GPa_ref'] else 'B=?'
    sig = f"σ={ref['sigma_mScm_ref']:.1f} mS/cm" if ref['sigma_mScm_ref'] else 'σ=new'
    print(f"  {s['material']}/{s['symmetry']}  ({s['mp_id']})  {b}  {sig}")

## 3 — Download POSCARs from Materials Project

In [ ]:
if RUN_DOWNLOAD:
    if MP_API_KEY == 'YOUR_MP_API_KEY_HERE':
        raise ValueError(
            'Set MP_API_KEY in cell 1 or as environment variable.\n'
            'Get your key at https://next-gen.materialsproject.org/api'
        )

    from mp_api.client import MPRester
    from pymatgen.io.vasp import Poscar

    mp_ids = [s['mp_id'] for s in VALIDATION_SYSTEMS]

    print(f'Querying Materials Project for {len(mp_ids)} structures...')
    with MPRester(MP_API_KEY) as mpr:
        docs = mpr.materials.summary.search(
            material_ids = mp_ids,
            fields       = ['material_id', 'formula_pretty', 'structure',
                            'symmetry', 'energy_per_atom', 'band_gap'],
            all_fields   = False,
        )

    # Index by mp_id
    docs_by_id = {str(doc.material_id): doc for doc in docs}

    for system in VALIDATION_SYSTEMS:
        mp_id    = system['mp_id']
        material = system['material']
        sym      = system['symmetry']

        doc = docs_by_id.get(mp_id)
        if doc is None:
            print(f'  [MISSING] {mp_id} not returned by MP API')
            continue
        if doc.structure is None:
            print(f'  [NO STRUCTURE] {mp_id}')
            continue

        # Write to database layout: database/<material>/<symmetry>/POSCAR
        sym_dir  = os.path.join(DATABASE_ROOT, material, sym)
        os.makedirs(sym_dir, exist_ok=True)
        poscar_dst = os.path.join(sym_dir, 'POSCAR')

        if not os.path.exists(poscar_dst) or FORCE_RERUN:
            poscar = Poscar(doc.structure)
            poscar.comment = f'{doc.formula_pretty} ({mp_id})'
            poscar.write_file(poscar_dst)

            # Write metadata alongside POSCAR (same convention as 00_MPquery.ipynb)
            metadata = {
                'material_id'   : str(mp_id),
                'formula'       : doc.formula_pretty,
                'symmetry'      : doc.symmetry.symbol if doc.symmetry else sym,
                'energy_per_atom': doc.energy_per_atom,
                'band_gap'      : doc.band_gap,
            }
            with open(os.path.join(sym_dir, 'metadata.json'), 'w') as fh:
                json.dump(metadata, fh, indent=2)

            print(f'  {material}/{sym}  ({mp_id})  →  {poscar_dst}')
        else:
            print(f'  {material}/{sym}  already exists — skipping')

    print('\nDownload complete.')
else:
    print('RUN_DOWNLOAD=False — skipping MP query.')
    print('Ensure POSCAR files exist in database/<material>/<symmetry>/POSCAR')

## 4 — Structure preparation (relaxation + phonons + elastic)

In [ ]:
if RUN_STRUCTURE:
    for system in VALIDATION_SYSTEMS:
        material = system['material']
        sym      = system['symmetry']
        sym_dir  = os.path.join(DATABASE_ROOT, material, sym)
        tag      = f'{material}/{sym}'

        if not os.path.exists(os.path.join(sym_dir, 'POSCAR')):
            print(f'  [SKIP] {tag}: POSCAR not found — run download step first')
            continue

        print(f'\n{"=" * 60}')
        print(f'  Structure prep: {tag}')
        print(f'{"=" * 60}')

        # Relaxation
        ret = run_relaxation_subprocess(sym_dir, device=DEVICE, force_rerun=FORCE_RERUN)
        if not ret:
            print(f'  [WARN] Relaxation failed for {tag}')

        # Phonons
        ret = run_phonons_subprocess(sym_dir, device=DEVICE, force_rerun=FORCE_RERUN)
        if not ret:
            print(f'  [WARN] Phonons failed for {tag}')

        # Elastic
        ret = run_elastic_subprocess(sym_dir, device=DEVICE, force_rerun=FORCE_RERUN)
        if not ret:
            print(f'  [WARN] Elastic failed for {tag}')

        print(f'  {tag}: structure prep complete')
else:
    print('RUN_STRUCTURE=False — skipping.')

## 5 — AIMD simulations + anharmonic VDOS

In [ ]:
if RUN_MD:
    for system in VALIDATION_SYSTEMS:
        material = system['material']
        sym      = system['symmetry']
        sym_dir  = os.path.join(DATABASE_ROOT, material, sym)
        temps    = system['temperatures']

        if not os.path.exists(os.path.join(sym_dir, 'relaxation', 'CONTCAR')):
            print(f'  [SKIP] {material}/{sym}: relaxation not complete')
            continue

        # Build AIMD supercells for all temperatures at once
        try:
            prepare_aimd_folders(sym_dir, temperatures=temps)
        except Exception as exc:
            print(f'  [WARN] AIMD prep failed: {exc}')
            continue

        for T in temps:
            tag = f'{material}/{sym}/{T}K'
            print(f'\n  Running MD: {tag}')

            script = os.path.join(PROJECT_ROOT, 'library', 'run_single_md.py')
            cmd    = [sys.executable, script, sym_dir, str(T), '--device', DEVICE]
            if FORCE_RERUN:
                cmd.append('--force')

            env = os.environ.copy()
            env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
            result = subprocess.run(cmd, env=env)

            if result.returncode != 0:
                print(f'  [FAIL] MD exited with code {result.returncode}')
                continue

            if RUN_ANHARMONIC:
                sim_dir = os.path.join(sym_dir, 'aimd', f'{T}K')
                try:
                    fe = compute_anharmonic_phonons_for_sim(
                        sim_dir,
                        temperatures    = HULL_TEMPERATURES,
                        force_recompute = FORCE_RERUN,
                    )
                    if fe is not None:
                        print(f'  Anharmonic VDOS cached → {sim_dir}/anharmonic_phonons/')
                except Exception as exc:
                    print(f'  [WARN] Anharmonic phonons failed: {exc}')
else:
    print('RUN_MD=False — skipping.')

## 6 — HeatUp stability assessment

In [ ]:
reports = {}

if RUN_STABILITY:
    for system in VALIDATION_SYSTEMS:
        material = system['material']
        sym      = system['symmetry']
        sym_dir  = os.path.join(DATABASE_ROOT, material, sym)
        tag      = f'{material}/{sym}'

        if not os.path.isdir(sym_dir):
            print(f'  [SKIP] {tag}: not found')
            continue

        print(f'\n{"=" * 60}')
        print(f'  HeatUp: {tag}')

        report = run_stability_pipeline(
            sym_dir                 = sym_dir,
            operating_T             = OPERATING_T,
            candidates_root         = CANDIDATES_DIR,
            database_root           = DATABASE_ROOT,
            temperatures            = HULL_TEMPERATURES,
            device                  = DEVICE,
            generate_missing_phases = True,
            force_rerun             = FORCE_RERUN,
            save_figure             = True,
        )
        reports[tag] = report

        print(f'  Overall: {report["overall"].upper()}')
        for flag in report.get('flags', []):
            print(f'    {flag}')
else:
    print('RUN_STABILITY=False — skipping.')

## 7 — Validation: Gate 1 (mechanical) vs literature

In [ ]:
print('=== Gate 1: Mechanical stability vs literature ===')
print(f'{"Material":<20} {"B_MACE":>10} {"B_ref":>8} {"G_MACE":>10} {"G_ref":>8} {"Born":>6} {"Status":>8}')
print('-' * 75)

mech_rows = []
for system in VALIDATION_SYSTEMS:
    tag = f"{system['material']}/{system['symmetry']}"
    rep = reports.get(tag, {})
    m   = rep.get('mechanical', {})
    ref = system['references']

    B_mace = m.get('bulk_modulus_GPa')
    G_mace = m.get('shear_modulus_GPa')
    born   = m.get('born_stable')
    status = m.get('status', '-')

    B_ref  = ref.get('B_GPa_ref')
    G_ref  = ref.get('G_GPa_ref')

    B_str = f'{B_mace:.0f}' if B_mace else '-'
    G_str = f'{G_mace:.0f}' if G_mace else '-'
    Br    = f'{B_ref:.0f}' if B_ref else 'N/A'
    Gr    = f'{G_ref:.0f}' if G_ref else 'N/A'
    bo    = '✓' if born else ('✗' if born is False else '-')

    print(f'{tag:<20} {B_str:>10} {Br:>8} {G_str:>10} {Gr:>8} {bo:>6} {status:>8}')

    # Percentage error if reference available
    if B_mace and B_ref:
        err_B = abs(B_mace - B_ref) / B_ref * 100
        err_G = abs(G_mace - G_ref) / G_ref * 100 if G_mace and G_ref else None
        mech_rows.append({
            'Material': tag, 'B_MACE': B_mace, 'B_ref': B_ref, 'err_B_%': err_B,
            'G_MACE': G_mace, 'G_ref': G_ref, 'err_G_%': err_G
        })

print()
print('  All values in GPa. B_ref / G_ref from DFT or experiment (see table in Section 2).')

if mech_rows:
    df_mech = pd.DataFrame(mech_rows)
    display(df_mech)

## 8 — Validation: Gate 2 (vibrational) vs literature

In [ ]:
print('=== Gate 2: Vibrational stability vs literature ===')
print(f'{"Material":<20} {"ζ (%)": >8} {"Exp stable":>12} {"Status":>8}')
print('-' * 55)

for system in VALIDATION_SYSTEMS:
    tag = f"{system['material']}/{system['symmetry']}"
    rep = reports.get(tag, {})
    v   = rep.get('vibrational', {})
    ref = system['references']

    zeta   = v.get('zero_window_frac')
    status = v.get('status', '-')
    exp_st = ref.get('vib_stable')

    z_str   = f'{zeta * 100:.2f}' if zeta is not None else '-'
    exp_str = '✓ stable' if exp_st is True else ('✗ unstable' if exp_st is False else 'N/A')

    # Agreement check
    heatup_stable = status == 'ok' or status == 'warn'
    agree = '✓' if (exp_st is None or exp_st == heatup_stable) else '✗'

    print(f'{tag:<20} {z_str:>8} {exp_str:>12} {status:>8}  {agree}')

print()
print('  ζ = fraction of VDOS within |ω| < 1 meV of zero.')
print('  Fail threshold: ζ > 8%.  AgI alpha-phase expected to fail above 420 K.')
print('  Li3PS4 and Na3PS4 expected to pass at 300 K.')

## 9 — Validation: Gate 3 (thermodynamic) vs literature

In [ ]:
print('=== Gate 3: Thermodynamic stability vs Materials Project ===')
print(f'{"Material":<20} {"ΔE_hull (meV)": >15} {"MP stable":>10} {"Status":>8}')
print('-' * 60)

for system in VALIDATION_SYSTEMS:
    tag = f"{system['material']}/{system['symmetry']}"
    rep = reports.get(tag, {})
    t   = rep.get('thermodynamic', {})
    ref = system['references']

    e_hull = t.get('e_above_hull_at_T_eV')
    status = t.get('status', '-')
    mp_st  = ref.get('hull_stable')

    e_str   = f'{e_hull * 1000:.1f}' if e_hull is not None else '-'
    mp_str  = '✓ stable' if mp_st is True else ('✗ unstable' if mp_st is False else 'N/A')
    agree   = '✓' if (mp_st is None or (mp_st == (status in ('ok','warn')))) else '✗'

    print(f'{tag:<20} {e_str:>15} {mp_str:>10} {status:>8}  {agree}')

print()
print(f'  Hull evaluated at T_op = {OPERATING_T:.0f} K.')
print(f'  Metastable threshold: {hcfg.THERMO_HULL_WARN_EV * 1000:.0f} meV/atom.')

## 10 — Validation: AIMD conductivity vs experiment

In [ ]:
print('=== AIMD ionic conductivity vs experiment ===')
print(f'{"Material":<20} {"T (K)":>6} {"Species":>8} {"D (cm²/s)":>14} {"σ_MACE (mS/cm)":>16} {"σ_exp (mS/cm)":>14}')
print('-' * 85)

for system in VALIDATION_SYSTEMS:
    material = system['material']
    sym      = system['symmetry']
    ref_sig  = system['references'].get('sigma_mScm_ref')
    if ref_sig is None:
        continue

    for T in system['temperatures']:
        sim_dir  = os.path.join(DATABASE_ROOT, material, sym, 'aimd', f'{T}K')
        analysis = load_analysis(sim_dir)
        if analysis is None:
            continue

        for sp, vals in analysis.get('diffusion', {}).items():
            D   = vals.get('diffusivity_cm2s', 0)
            sig = vals.get('conductivity_mScm', 0)
            if D < 1e-15:
                continue

            exp_str = f'{ref_sig:.2f}' if ref_sig else 'N/A'
            ratio   = sig / ref_sig if ref_sig and sig > 0 else None
            ratio_str = f'×{ratio:.1f}' if ratio else ''

            print(f'{material}/{sym}      {T:>6} {sp:>8} {D:>14.3e} {sig:>16.4f} {exp_str:>14}  {ratio_str}')

print()
print('  Note: AIMD conductivities at finite T are compared with room-temperature experiments.')
print('  Arrhenius extrapolation to 300 K would be needed for a rigorous comparison.')

## 11 — Comprehensive validation figure (paper Figure 6)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))

# ── (a) Elastic moduli: MACE vs literature ────────────────────────────────────
ax = axes[0]
B_mace_vals, B_ref_vals, labels = [], [], []
G_mace_vals, G_ref_vals         = [], []

for system in VALIDATION_SYSTEMS:
    tag = f"{system['material']}/{system['symmetry']}"
    rep = reports.get(tag, {})
    m   = rep.get('mechanical', {})
    ref = system['references']
    if m.get('bulk_modulus_GPa') and ref.get('B_GPa_ref'):
        B_mace_vals.append(m['bulk_modulus_GPa'])
        B_ref_vals.append(ref['B_GPa_ref'])
        G_mace_vals.append(m['shear_modulus_GPa'])
        G_ref_vals.append(ref['G_GPa_ref'])
        labels.append(system['material'])

if B_mace_vals:
    ax.scatter(B_ref_vals, B_mace_vals, s=80, color='#0072B2',
               label='Bulk modulus B', zorder=3, marker='o')
    ax.scatter(G_ref_vals, G_mace_vals, s=80, color='#E69F00',
               label='Shear modulus G', zorder=3, marker='s')
    for i, lab in enumerate(labels):
        ax.annotate(lab, (B_ref_vals[i], B_mace_vals[i]),
                    fontsize=6, xytext=(3, 3), textcoords='offset points')

    all_vals = B_mace_vals + B_ref_vals + G_mace_vals + G_ref_vals
    lim = [0, max(all_vals) * 1.15]
    ax.plot(lim, lim, 'k--', lw=0.8, alpha=0.5, label='1:1 line')
    ax.set_xlim(lim); ax.set_ylim(lim)

ax.set_xlabel('Reference (GPa)', fontsize=9)
ax.set_ylabel('MACE-MP (GPa)', fontsize=9)
ax.set_title('(a) Elastic moduli', fontsize=9, fontweight='bold')
ax.legend(frameon=False, fontsize=7)

# ── (b) Hull distance vs temperature ─────────────────────────────────────────
ax = axes[1]
T_grid  = np.array(HULL_TEMPERATURES, dtype=float)
colours = ['#0072B2','#E69F00','#009E73','#D55E00','#CC79A7']

for i, system in enumerate(VALIDATION_SYSTEMS):
    tag = f"{system['material']}/{system['symmetry']}"
    rep = reports.get(tag, {})
    t   = rep.get('thermodynamic', {})
    if not t.get('hull_results'):
        continue
    valid = [(r['T'], r['e_above_hull_eV_atom'])
             for r in t['hull_results'] if r.get('e_above_hull_eV_atom') is not None]
    if not valid:
        continue
    T_arr = np.array([v[0] for v in valid])
    E_meV = np.array([v[1] for v in valid]) * 1000
    ax.plot(T_arr, E_meV, color=colours[i % len(colours)],
            lw=1.3, label=system['material'])

ax.axhline(0,                              color='green',  lw=0.8, ls='--', alpha=0.7)
ax.axhline(hcfg.THERMO_HULL_WARN_EV*1000, color='orange', lw=0.8, ls='--', alpha=0.7)
ax.axvline(OPERATING_T,                    color='gray',   lw=0.8, ls=':',  alpha=0.6)
ax.set_xlabel('Temperature (K)', fontsize=9)
ax.set_ylabel('ΔE above hull (meV/atom)', fontsize=9)
ax.set_title('(b) Temperature-dependent hull distance', fontsize=9, fontweight='bold')
ax.legend(frameon=False, fontsize=7)
ax.set_xlim(0, max(HULL_TEMPERATURES))

# ── (c) AIMD conductivity vs experiment ──────────────────────────────────────
ax = axes[2]
mace_sigs, exp_sigs, cond_labels = [], [], []

for system in VALIDATION_SYSTEMS:
    material = system['material']
    sym      = system['symmetry']
    ref_sig  = system['references'].get('sigma_mScm_ref')
    if ref_sig is None:
        continue
    for T in system['temperatures']:
        sim_dir  = os.path.join(DATABASE_ROOT, material, sym, 'aimd', f'{T}K')
        analysis = load_analysis(sim_dir)
        if analysis is None:
            continue
        for sp, vals in analysis.get('diffusion', {}).items():
            sig = vals.get('conductivity_mScm', 0)
            if sig > 0.01:  # filter non-mobile species
                mace_sigs.append(sig)
                exp_sigs.append(ref_sig)
                cond_labels.append(f'{material}({sp})@{T}K')

if mace_sigs:
    ax.scatter(exp_sigs, mace_sigs, s=80, color='#0072B2', zorder=3)
    for i, lab in enumerate(cond_labels):
        ax.annotate(lab, (exp_sigs[i], mace_sigs[i]),
                    fontsize=6, xytext=(3, 3), textcoords='offset points')
    all_c = mace_sigs + exp_sigs
    lim_c = [min(all_c)*0.5, max(all_c)*2]
    ax.plot(lim_c, lim_c, 'k--', lw=0.8, alpha=0.5)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlim(lim_c); ax.set_ylim(lim_c)
else:
    ax.text(0.5, 0.5, 'Run AIMD to populate', transform=ax.transAxes,
            ha='center', va='center', fontsize=9, color='gray')

ax.set_xlabel('Experimental σ (mS/cm)', fontsize=9)
ax.set_ylabel('MACE AIMD σ (mS/cm)', fontsize=9)
ax.set_title('(c) Ionic conductivity', fontsize=9, fontweight='bold')

fig.suptitle('HeatUp validation against experimental and DFT reference data',
             fontsize=10, fontweight='bold')
plt.tight_layout()
plt.savefig('heatup_validation_figure.pdf', dpi=300, bbox_inches='tight')
plt.show()
print('Saved: heatup_validation_figure.pdf')

## 12 — Save full validation report

In [ ]:
validation_output = {
    'systems'   : VALIDATION_SYSTEMS,
    'reports'   : {k: {ck: cv for ck, cv in v.items()
                       if ck not in ('vibrational',)}
                   for k, v in reports.items()},
    'operating_T': OPERATING_T,
    'mace_model' : MACE_MODEL_PATH,
}

with open('validation_results.json', 'w') as fh:
    json.dump(validation_output, fh, indent=2, default=str)
print('Saved: validation_results.json')

print('\n=== VALIDATION SUMMARY ===')
for system in VALIDATION_SYSTEMS:
    tag = f"{system['material']}/{system['symmetry']}"
    rep = reports.get(tag, {})
    print(f'  {tag:<25}  overall={rep.get("overall", "N/A").upper():<8}  '
          f'mech={rep.get("mechanical",{}).get("status","-"):<8}  '
          f'vib={rep.get("vibrational",{}).get("status","-"):<8}  '
          f'thermo={rep.get("thermodynamic",{}).get("status","-")}')